# EMBED preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.embed import Embed, csv_save_path
from utils.preprocessing import (
    CRANIAL_CAUDAL, MEDIOLATERAL_OBLIQUE, UNKNOWN, birads_assessment_reverse,
    csv_column_cleaning, dview, get_value, isna_v2, laterality,
)

ds = Embed()
ds.set_dataset_name("embed")
print(ds.metadata_df.shape, ds.clinical_df.shape, ds.imgs_size_df.shape)
ds.clinical_df.head()

### Step 1 — drop spot-magnification images (we want the full breast view)

In [ ]:
ds.metadata_df = ds.metadata_df[ds.metadata_df["spot_mag"].isna()]
ds.metadata_df.shape

### Step 2/3 — drop unassessed rows ('A'/'X'), fold BI-RADS 6 ('K') into BI-RADS 5 ('M')

In [ ]:
ds.clinical_df = ds.clinical_df[(ds.clinical_df["asses"] != "A") & (ds.clinical_df["asses"] != "X")]
ds.clinical_df["asses"] = ds.clinical_df["asses"].replace("K", "M")
ds.clinical_df["asses"].value_counts(dropna=False)

### Step 4/5 — keep only the columns needed from metadata and clinical sheets

In [ ]:
ds.metadata_df = ds.metadata_df[[
    "empi_anon", "acc_anon", "study_date_anon", "FinalImageType", "ImageLateralityFinal",
    "ViewPosition", "anon_dicom_path", "num_roi", "ROI_coords", "SRC_DST", "match_level",
    "Manufacturer", "ManufacturerModelName", "SeriesDescription",
]]
ds.clinical_df = ds.clinical_df[[
    "massshape", "massmargin", "massdens", "calcfind", "calcdistri", "calcnumber", "otherfind",
    "implanfind", "side", "location", "depth", "asses", "desc", "tissueden", "GENDER_DESC",
    "ETHNICITY_DESC", "age_at_study", "empi_anon", "acc_anon", "study_date_anon", "recc",
    "total_L_find", "total_R_find",
]]
ds.metadata_df.shape, ds.clinical_df.shape

### Step 6 — drop unassessed/biopsy-already-performed recommendations

In [ ]:
ds.clinical_df = ds.clinical_df[(ds.clinical_df["recc"] != "Z") & (ds.clinical_df["recc"] != "&")]
ds.clinical_df["recc"] = ds.clinical_df["recc"].apply(lambda x: ds.clean_recc(x) if not isna_v2(x) else x)
ds.clinical_df["recc"].value_counts(dropna=False).head()

### Step 7 — filter for women only (for the moment)

In [ ]:
ds.clinical_df = ds.clinical_df[ds.clinical_df["GENDER_DESC"] == "Female"]
ds.clinical_df.shape

### Step 8 — drop duplicates, then parse the study dates used to join the two sheets

In [ ]:
ds.metadata_df = ds.metadata_df.drop_duplicates()
ds.clinical_df = ds.clinical_df.drop_duplicates()
ds.metadata_df["study_date_anon"] = pd.to_datetime(ds.metadata_df["study_date_anon"], errors="coerce")
ds.clinical_df["study_date_anon"] = pd.to_datetime(ds.clinical_df["study_date_anon"], errors="coerce")
ds.metadata_df.shape, ds.clinical_df.shape

### Step 9/10 — merge metadata with clinical on exam id, then keep rows whose finding side matches the image laterality

In [ ]:
df_merged = pd.merge(ds.metadata_df, ds.clinical_df, on=["acc_anon", "empi_anon", "study_date_anon"], how="inner")
df_merged = df_merged.loc[
    (df_merged.side == df_merged.ImageLateralityFinal) | (df_merged.side == "B") | (pd.isna(df_merged.side))
]
df_merged.shape

### Step 11 — build the clinical column-renaming map and the coded-value legend from EMBED's data dictionary

In [ ]:
clinical_column_renaming = ds.construct_column_renaming()
legend_dict = ds.construct_legend_dict()
list(legend_dict.keys())[:5]

### Step 12 — manually decode `tissueden` (it uses a different code shape than the rest of the legend)

In [ ]:
df_merged["tissueden"] = df_merged["tissueden"].apply(
    lambda x: legend_dict["tissueden"][str(int(float(x)))] if not isna_v2(x) else None
)
df_merged["tissueden"].value_counts(dropna=False)

### Step 13 — decode the remaining coded columns via the legend, then rename and clean all column names

In [ ]:
manual_adjustments = ["tissueden"]
for header in legend_dict:
    if header in df_merged.columns and header not in manual_adjustments:
        df_merged[header] = df_merged[header].apply(lambda x: ds.apply_legend_dict(x, header, legend_dict))

df_merged = df_merged.rename(columns=clinical_column_renaming)
df_merged.columns = csv_column_cleaning(list(df_merged.columns))
df_merged.columns.tolist()[:10]

### Step 14 — collapse the unhelpful ethnicity edge cases to `unknown`

In [ ]:
ethnicity_edge_cases_dict = {
    "Patient Declines": UNKNOWN,
    "Not Recorded": UNKNOWN,
    "Unknown, Unavailable or Unreported": UNKNOWN,
}
df_merged["ethnicity desc"] = df_merged["ethnicity desc"].apply(lambda x: ethnicity_edge_cases_dict.get(x, x))
df_merged["ethnicity desc"].value_counts(dropna=False)

### Step 15 — map laterality/view position, keep only CC/MLO views

In [ ]:
df_merged["image laterality final"] = df_merged["image laterality final"].apply(lambda x: get_value(x, laterality))
df_merged["view position"] = df_merged["view position"].apply(lambda x: get_value(x, dview) if not isna_v2(x) else None)
df_merged = df_merged[
    df_merged["view position"].isin([MEDIOLATERAL_OBLIQUE, CRANIAL_CAUDAL])
].reset_index(drop=True)
df_merged["view position"].value_counts(dropna=False)

### Step 16 — cast age to int

In [ ]:
df_merged["age at study"] = df_merged["age at study"].apply(lambda x: int(x) if not isna_v2(x) else None)
df_merged["age at study"].describe()

### Step 17 — attach original/resized image dimensions (needed to scale ROI segmentations later)

In [ ]:
df_merged["image id"] = df_merged["anon dicom path"].apply(lambda x: os.path.basename(x)[:-4])
ds.imgs_size_df["image id"] = ds.imgs_size_df["image path"].apply(lambda x: os.path.basename(x)[:-4])
out_df = pd.merge(df_merged, ds.imgs_size_df, on=["image id"], how="inner")
out_df.shape

### Step 18 — normalize column names to snake_case, drop rows missing a series description, keep only 2D mammograms

In [ ]:
out_df = out_df.rename(columns=lambda c: c.strip().replace(" ", "_").replace("-", "_").lower())
out_df = out_df[out_df["series_description"].notna()]
out_df = out_df[out_df["final_image_type"] == "2D"]
out_df.shape

### Step 19 — rank candidate findings per image and keep a single primary finding

In [ ]:
out_df["_birads_rank"] = out_df["assessment_bi_rads"].map(birads_assessment_reverse).fillna(1).astype(int)
out_df["_roi_rank"] = out_df["roi_coords"].apply(ds.roi_priority)
out_df["_side_rank"] = out_df["side"].apply(ds.side_priority)
out_df["_lesion_type_rank"] = out_df.apply(ds.lesion_type_priority, axis=1)
out_df["_descriptor_rank"] = out_df.apply(ds.descriptor_priority, axis=1)

out_df = (
    out_df.sort_values(
        by=["empi_anon", "acc_anon", "image_id", "_birads_rank", "_roi_rank", "_side_rank",
            "_descriptor_rank", "_lesion_type_rank"],
        ascending=[True, True, True, False, False, False, False, False],
        kind="mergesort",
    )
    .drop_duplicates(subset=["empi_anon", "acc_anon", "image_id"], keep="first")
    .drop(columns=["_birads_rank", "_roi_rank", "_side_rank", "_lesion_type_rank", "_descriptor_rank"])
    .reset_index(drop=True)
)
out_df.shape

### Step 20 — drop rows with neither a finding side nor an ROI (laterality can't be determined)

In [ ]:
out_df = out_df[~(out_df["side"].isna() & out_df["roi_coords"].isna())]
out_df.shape

## Build one exam record

In [ ]:
grouped = out_df.groupby(by=["empi_anon", "acc_anon", "image_id"], sort=False)
_, group = next(iter(grouped))
exams = ds.process_patient_exam_img(group)
exams

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"(patient, exam, image) groups: {grouped.ngroups} | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))